In [1]:
import getml
import mlflow
import getml_mlflow

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
getml_mlflow.autolog()

In [3]:
getml.set_project("interstate94")

Output()

Connected to project 'interstate94'.

In [4]:
traffic = getml.datasets.load_interstate94(roles=False, units=False)

In [5]:
traffic.set_role("ds", getml.data.roles.time_stamp)
traffic.set_role("holiday", getml.data.roles.categorical)
traffic.set_role("traffic_volume", getml.data.roles.target)

In [6]:
split = getml.data.split.time(traffic, "ds", test=getml.data.time.datetime(2018, 3, 15))

In [7]:
time_series = getml.data.TimeSeries(
    population=traffic,
    split=split,
    time_stamps="ds",
    horizon=getml.data.time.hours(1),
    memory=getml.data.time.days(7),
    lagged_targets=True,
)

pipe = getml.pipeline.Pipeline(
    tags=["memory: 7d", "horizon: 1h", "fast_prop"],
    data_model=time_series.data_model,
    preprocessors=[getml.preprocessors.Seasonal()],
    feature_learners=[
        getml.feature_learning.FastProp(
            loss_function=getml.feature_learning.loss_functions.SquareLoss,
            num_threads=1,
            num_features=20,
        )
    ],
    predictors=[getml.predictors.XGBoostRegressor()],
)
pipe

Pipeline(data_model='population',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=False,
         loss_function='SquareLoss',
         peripheral=['traffic'],
         predictors=['XGBoostRegressor'],
         preprocessors=['Seasonal'],
         share_selected_features=0.5,
         tags=['memory: 7d', 'horizon: 1h', 'fast_prop'])

In [8]:
fit1 = pipe.fit(time_series.train)
print(fit1.id, pipe.id)

2025-01-28 19:05:45,263 WARNING getML: Engine metrics are available in the Enterprise edition. Visit https://getml.com/latest/enterprise/ for more information
2025-01-28 19:05:45,271 WARNING getML: Engine metrics are available in the Enterprise edition. Visit https://getml.com/latest/enterprise/ for more information


Checking data model...

Output()

OK.

Output()

Trained pipeline.

2025/01/28 19:05:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run fit at: http://localhost:5000/#/experiments/844494434965818253/runs/7ddf1883b18541b798745788d2ddb4ad.
2025/01/28 19:05:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.
2025/01/28 19:05:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run Pipeline-Y6WffS at: http://localhost:5000/#/experiments/844494434965818253/runs/507ab6ed28c2452481b21274a7f0b7e4.
2025/01/28 19:05:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


Time taken: 0:00:08.388892.

Y6WffS Y6WffS


In [9]:
fit2 = pipe.fit(time_series.train)
print(fit2.id, fit1.id, pipe.id)

2025-01-28 19:05:54,066 WARNING getML: Engine metrics are available in the Enterprise edition. Visit https://getml.com/latest/enterprise/ for more information
2025-01-28 19:05:54,074 WARNING getML: Engine metrics are available in the Enterprise edition. Visit https://getml.com/latest/enterprise/ for more information


Checking data model...

Output()

OK.

Output()

Trained pipeline.

2025/01/28 19:05:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run fit at: http://localhost:5000/#/experiments/844494434965818253/runs/7ac345185a6c46edab2bb7f26313d89f.
2025/01/28 19:05:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.
2025/01/28 19:05:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run Pipeline-emTA9s at: http://localhost:5000/#/experiments/844494434965818253/runs/c6027b5f3aa24782939d57befadb53ba.
2025/01/28 19:05:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


Time taken: 0:00:00.254767.

emTA9s emTA9s emTA9s


In [10]:
pipe.score(time_series.test)

Output()

2025/01/28 19:05:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run score at: http://localhost:5000/#/experiments/844494434965818253/runs/30f4f0474719433e9d8f3c3ec4a5af20.
2025/01/28 19:05:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


,date time,set used,target,mae,rmse,rsquared
0,2025-01-28 19:05:54,train,traffic_volume,200.4302,299.2045,0.9768
1,2025-01-28 19:05:54,test,traffic_volume,179.9515,269.631,0.9816


In [11]:
pipe.id

'emTA9s'

In [12]:
# mlflow.data.dataset_source_registry.resolve_dataset_source("traffic.train.parquet")

In [13]:
ds = mlflow.data.dataset_source_registry.resolve_dataset_source("traffic.train.parquet")
print(ds)
print(ds.to_dict())
print(mlflow.get_artifact_uri("traffic.train.parquet"))

/home/manuel/Projects/github/getml-mlflow/.venv/lib/python3.11/site-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'traffic.train.parquet'. Exception: 
  return _dataset_source_registry.resolve(
/home/manuel/Projects/github/getml-mlflow/.venv/lib/python3.11/site-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


{'uri': 'traffic.train.parquet'}
mlflow-artifacts:/0/10eaa89156444e25920a87c24c2f6988/artifacts/traffic.train.parquet


In [14]:
# run = mlflow.last_active_run()
# run.info.run_name = "ÄÄÄ"

In [15]:
pipe._mlflow_run_info

<RunInfo: artifact_uri='mlflow-artifacts:/844494434965818253/c6027b5f3aa24782939d57befadb53ba/artifacts', end_time=None, experiment_id='844494434965818253', lifecycle_stage='active', run_id='c6027b5f3aa24782939d57befadb53ba', run_name='Pipeline-Y6WffS', run_uuid='c6027b5f3aa24782939d57befadb53ba', start_time=1738087554041, status='RUNNING', user_id='unknown'>